In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Attempt-1 (With rhyming scheme)

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import re
import json
from collections import defaultdict
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

# Custom dataset for lyrics generation
class LyricsDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.inputs = []
        self.max_length = max_length
        
        for _, row in data.iterrows():
            # Create prompt with audio features
            prompt = f"[FEATURES] duration_ms: {row['duration_ms']}, tempo: {row['tempo']}, "
            prompt += f"energy: {row['energy']}, loudness: {row['loudness']}, "
            prompt += f"danceability: {row['danceability']}, speechiness: {row['speechiness']}, "
            prompt += f"acousticness: {row['acousticness']}, instrumentalness: {row['instrumentalness']}, "
            prompt += f"liveness: {row['liveness']}, valence: {row['valence']}, "
            prompt += f"Sentiments: {row['Sentiment']}, "
            
            # # Add rhyme scheme if available
            # if 'rhyme_scheme' in row and not pd.isna(row['rhyme_scheme']):
            #     prompt += f"rhyme_scheme: {row['rhyme_scheme']}, "
            
            prompt += "[LYRICS] "
            
            # Complete text (prompt + lyrics)
            complete_text = prompt + row['Hindi Lyrics']
            
            encodings_dict = tokenizer(complete_text, 
                                      truncation=True,
                                      max_length=self.max_length, 
                                      padding="max_length")
            
            self.inputs.append({
                'input_ids': torch.tensor(encodings_dict['input_ids']),
                'attention_mask': torch.tensor(encodings_dict['attention_mask']),
                'labels': torch.tensor(encodings_dict['input_ids'])
            })
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return self.inputs[idx]

# Custom trainer with rhyme-aware loss
class RhymeAwareTrainer:
    def __init__(self, model, tokenizer, train_dataloader, val_dataloader=None, 
                 rhyme_weight=0.5, num_epochs=5, lr=5e-5, warmup_steps=0):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.rhyme_weight = rhyme_weight
        self.num_epochs = num_epochs
        
        # Setup optimizer and scheduler
        self.optimizer = AdamW(model.parameters(), lr=lr)
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer, 
            num_warmup_steps=warmup_steps,
            num_training_steps=len(train_dataloader) * num_epochs
        )
        
        # Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        
        # Hindi phonetic mappings for rhyme detection
        self.hindi_endings = self._setup_hindi_phonetic_mappings()
    
    def _setup_hindi_phonetic_mappings(self):
        """Simple phonetic mappings for Hindi transliteration endings"""
        # This is a simplified approach - a more comprehensive solution would use 
        # proper phonetic analysis libraries for Hindi transliteration
        endings = {
            'a': ['a', 'ah', 'aa'],
            'e': ['e', 'ey', 'ay'],
            'i': ['i', 'ee', 'y'],
            'o': ['o', 'oh', 'ow'],
            'u': ['u', 'oo'],
            'an': ['an', 'un', 'on'],
            'in': ['in', 'een'],
            'ar': ['ar', 'er', 'ir', 'ur'],
            # Add more phonetic groups as needed
        }
        return endings
    
    def _calculate_rhyme_penalty(self, generated_text):
        """Calculate penalty for lack of rhyming in generated text"""
        # Extract only the lyrics part (after [LYRICS])
        lyrics_match = re.search(r'\[LYRICS\](.*)', generated_text, re.DOTALL)
        if not lyrics_match:
            return 0.0
        
        lyrics = lyrics_match.group(1).strip()
        
        # Split lyrics into lines
        lines = [line.strip() for line in lyrics.split('\n') if line.strip()]
        if len(lines) <= 1:
            return 0.0
        
        # Extract last word of each line
        last_words = []
        for line in lines:
            words = word_tokenize(line.lower())
            if words:
                last_words.append(words[-1])
            else:
                last_words.append("")
        
        # Check for rhyming patterns
        rhyme_score = 0.0
        expected_patterns = [
            # AABB pattern
            [(0,2), (1,3)],
            # ABAB pattern
            [(0,2), (1,3)],
            # AAAA pattern
            [(0,1), (1,2), (2,3)]
        ]
        
        # Simple check for ending similarity
        def words_rhyme(word1, word2):
            # Check if last 2-3 characters match for basic rhyming
            if len(word1) >= 2 and len(word2) >= 2:
                # Check for exact ending matches
                if word1[-2:] == word2[-2:]:
                    return True
                
                # Check against phonetic mapping
                for ending_group in self.hindi_endings.values():
                    word1_matches = any(word1.endswith(end) for end in ending_group)
                    word2_matches = any(word2.endswith(end) for end in ending_group)
                    if word1_matches and word2_matches:
                        return True
            return False
        
        # Check each expected pattern
        for pattern in expected_patterns:
            pattern_score = 0
            for i, j in pattern:
                if i < len(last_words) and j < len(last_words):
                    if words_rhyme(last_words[i], last_words[j]):
                        pattern_score += 1
                        
            # Normalize by number of comparisons
            if pattern_score > 0:
                pattern_score /= len(pattern)
                rhyme_score = max(rhyme_score, pattern_score)
        
        # Return penalty (inverse of score)
        return 1.0 - rhyme_score
    
    def train(self):
        self.model.train()
        total_loss = 0
        
        for epoch in range(self.num_epochs):
            print(f"Starting epoch {epoch+1}/{self.num_epochs}")
            
            for batch_idx, batch in enumerate(self.train_dataloader):
                # Move batch to device
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                # Forward pass
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                # Standard language modeling loss
                lm_loss = outputs.loss
                
                # Custom rhyme loss calculation (basic implementation)
                # In practice, you'd need a more sophisticated approach
                if self.rhyme_weight > 0:
                    rhyme_loss = torch.tensor(0.0).to(self.device)
                    
                    # For a small batch sample, decode and evaluate rhyming
                    if batch_idx % 100 == 0:
                        sample_idx = np.random.randint(0, input_ids.shape[0])
                        sample_text = self.tokenizer.decode(input_ids[sample_idx])
                        rhyme_penalty = self._calculate_rhyme_penalty(sample_text)
                        rhyme_loss = torch.tensor(rhyme_penalty).to(self.device)
                        
                        print(f"Sample rhyme penalty: {rhyme_penalty:.4f}")
                else:
                    rhyme_loss = torch.tensor(0.0).to(self.device)
                
                # Combined loss
                loss = lm_loss + (self.rhyme_weight * rhyme_loss)
                
                # Backward pass and optimization
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
                
                total_loss += loss.item()
                
                if batch_idx % 50 == 0:
                    print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}")
        
        return total_loss / (self.num_epochs * len(self.train_dataloader))
    
    def save_model(self, output_dir):
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")
    
    def evaluate(self):
        if not self.val_dataloader:
            return None
            
        self.model.eval()
        eval_loss = 0
        
        with torch.no_grad():
            for batch in self.val_dataloader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                eval_loss += outputs.loss.item()
                
        return eval_loss / len(self.val_dataloader)

# Main training function
def train_lyrics_model(data_path, output_dir, model_name="gpt2", 
                       batch_size=4, epochs=3, rhyme_weight=0.5):
    """Train GPT-2 model on lyrics dataset with rhyme-aware loss"""
    
    # Load and prepare data
    data = pd.read_csv(data_path)
    
    # Make sure all required columns exist
    required_columns = [
        'duration_ms', 'tempo', 'energy', 'loudness', 'danceability',
        'speechiness', 'acousticness', 'instrumentalness', 'liveness',
        'valence', 'Hindi Lyrics'
    ]
    
    for col in required_columns:
        if col not in data.columns:
            print(f"Warning: Missing column {col}")
            if col != 'Hindi Lyrics':  # Lyrics column is essential
                data[col] = 0  # Fill with defaults for missing audio features
    
    # Split data
    train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)
    
    # Load tokenizer and model
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name)
    
    # Add special tokens
    special_tokens = {
        'pad_token': '[PAD]',
        'additional_special_tokens': ['[FEATURES]', '[LYRICS]'],
    }
    tokenizer.add_special_tokens(special_tokens)
    model.resize_token_embeddings(len(tokenizer))
    
    # Create datasets
    train_dataset = LyricsDataset(train_data, tokenizer)
    val_dataset = LyricsDataset(val_data, tokenizer)
    
    # Create dataloaders
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size
    )
    
    # Set up trainer
    trainer = RhymeAwareTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        rhyme_weight=rhyme_weight,
        num_epochs=epochs
    )
    
    # Train model
    train_loss = trainer.train()
    print(f"Training completed with average loss: {train_loss:.4f}")
    
    # Evaluate model
    eval_loss = trainer.evaluate()
    if eval_loss:
        print(f"Evaluation loss: {eval_loss:.4f}")
    
    # Save model
    trainer.save_model(output_dir)
    
    return model, tokenizer

# Function to generate lyrics based on audio features
def generate_lyrics(model, tokenizer, audio_features, max_length=200, 
                   temperature=0.8, top_k=50, top_p=0.95):
    """Generate lyrics based on provided audio features"""
    
    # Create prompt from features
    prompt = "[FEATURES] "
    for key, value in audio_features.items():
        prompt += f"{key}: {value}, "
    prompt += "[LYRICS]"
    
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    # Move to model's device
    device = model.device
    input_ids = input_ids.to(device)
    
    # Set attention mask
    attention_mask = torch.ones(input_ids.shape, device=device)
    
    # Generate lyrics
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )
    
    # Decode output
    generated_text = tokenizer.decode(output[0], skip_special_tokens=False)
    
    # Extract only the lyrics part
    lyrics_match = re.search(r'\[LYRICS\](.*)', generated_text, re.DOTALL)
    if lyrics_match:
        return lyrics_match.group(1).strip()
    else:
        return generated_text

# Example usage
if __name__ == "__main__":
    # This would be your main execution code
    data_path = "/kaggle/input/final-hindi-dataset/hindi_songs_with_sentiment.csv"
    output_dir = "./hindi_lyrics_gpt2"
    
    # Train model
    model, tokenizer = train_lyrics_model(
        data_path=data_path,
        output_dir=output_dir,
        rhyme_weight=0.5,  # Weight for the rhyme component of the loss
        epochs=3
    )
    
    # Example of generating lyrics
    sample_features = {
        "duration_ms": 230000,
        "tempo": 96,
        "energy": 0.8,
        "loudness": -6.2,
        "danceability": 0.75,
        "speechiness": 0.04,
        "acousticness": 0.1,
        "instrumentalness": 0.0,
        "liveness": 0.2,
        "valence": 0.65,
        "rhyme_scheme": "AABB"
    }
    
    generated_lyrics = generate_lyrics(model, tokenizer, sample_features)
    print("Generated Lyrics:")
    print(generated_lyrics)

In [ ]:
import torch
import numpy as np
import re
import json
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

class HindiRhymeEvaluator:
    """Evaluates the quality of rhyming in Hindi lyrics (transliterated to English)"""
    
    def __init__(self):
        # Define phonetic groups for Hindi transliteration
        self.phonetic_groups = {
            # Vowel endings
            'aa': ['a', 'aa', 'ah'],
            'ee': ['i', 'ee', 'y', 'ey', 'i'],
            'oo': ['u', 'oo'],
            'ai': ['ai', 'ae', 'ayi'],
            'ei': ['e', 'ey', 'ei'],
            'au': ['au', 'ow'],
            
            # Consonant-vowel combinations
            'na': ['na', 'nah'],
            'ni': ['ni', 'nee'],
            'ne': ['ne', 'ney'],
            'no': ['no', 'noh'],
            'nu': ['nu', 'noo'],
            
            'ta': ['ta', 'tah', 'tha', 'thah'],
            'ti': ['ti', 'tee', 'thi', 'thee'],
            'te': ['te', 'they', 'the', 'they'],
            'to': ['to', 'toh', 'tho', 'thoh'],
            'tu': ['tu', 'too', 'thu', 'thoo'],
            
            'ra': ['ra', 'rah'],
            'ri': ['ri', 'ree'],
            're': ['re', 'rey'],
            'ro': ['ro', 'roh'],
            'ru': ['ru', 'roo'],
            
            # Add more phonetic groups as needed
            'ya': ['ya', 'yah'],
            'ma': ['ma', 'mah'],
            'la': ['la', 'lah'],
            'sa': ['sa', 'sah', 'sha', 'shah'],
            'ka': ['ka', 'kah', 'qa', 'qah'],
            'pa': ['pa', 'pah'],
            'ja': ['ja', 'jah', 'za', 'zah'],
            'ba': ['ba', 'bah'],
            'da': ['da', 'dah'],
        }
        
        # Create reverse mapping for faster lookup
        self.ending_to_group = {}
        for group, endings in self.phonetic_groups.items():
            for ending in endings:
                self.ending_to_group[ending] = group
    
    def words_rhyme(self, word1, word2):
        """Check if two words rhyme based on Hindi transliteration rules"""
        word1 = word1.lower()
        word2 = word2.lower()
        
        # Words are identical
        if word1 == word2:
            return 1.0
        
        # Check for exact ending match (2-3 characters)
        for length in [3, 2]:
            if len(word1) >= length and len(word2) >= length and word1[-length:] == word2[-length:]:
                return 0.9
        
        # Check for phonetic group match
        # Extract last syllable (approximation for Hindi transliteration)
        w1_last_syl = self._extract_last_syllable(word1)
        w2_last_syl = self._extract_last_syllable(word2)
        
        # Check if both syllables belong to the same phonetic group
        if w1_last_syl in self.ending_to_group and w2_last_syl in self.ending_to_group:
            if self.ending_to_group[w1_last_syl] == self.ending_to_group[w2_last_syl]:
                return 0.8
        
        # Check for partial matches
        if any(w1_last_syl.endswith(end) and w2_last_syl.endswith(end) for end in ['a', 'e', 'i', 'o', 'u']):
            return 0.6
            
        return 0.0
    
    def _extract_last_syllable(self, word):
        """Extract last syllable from a word (simplified for Hindi transliteration)"""
        if len(word) <= 2:
            return word
            
        # Look for vowel+consonant patterns from the end
        vowels = 'aeiou'
        consonants = 'bcdfghjklmnpqrstvwxyz'
        
        for i in range(1, min(4, len(word))):
            if word[-i] in vowels:
                # Found a vowel, extract from here to end
                for j in range(i+1, min(i+3, len(word)+1)):
                    if j == len(word) or word[-(j)] in consonants:
                        return word[-j:]
        
        # Default to last 2 characters
        return word[-2:]
    
    def evaluate_rhyme_scheme(self, lyrics):
        """Evaluate the rhyme scheme of lyrics"""
        # Split lyrics into lines
        lines = [line.strip() for line in lyrics.split('\n') if line.strip()]
        if len(lines) <= 1:
            return {
                'score': 0.0,
                'scheme': '',
                'pattern': []
            }
        
        # Extract last word of each line
        last_words = []
        for line in lines:
            words = word_tokenize(line.lower())
            if words:
                last_words.append(words[-1])
            else:
                last_words.append("")
        
        # Build rhyme matrix (showing which lines rhyme with which)
        n = len(last_words)
        rhyme_matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(i, n):
                if i == j:
                    rhyme_matrix[i][j] = 1.0
                else:
                    rhyme_score = self.words_rhyme(last_words[i], last_words[j])
                    rhyme_matrix[i][j] = rhyme_score
                    rhyme_matrix[j][i] = rhyme_score
        
        # Detect rhyme scheme
        # For simplicity, we'll consider a threshold-based approach
        threshold = 0.7  # Words with rhyme score above this are considered rhyming
        
        # Convert to letter-based scheme (A, B, C, etc.)
        scheme = [''] * n
        next_letter = 0
        letter_map = {}
        
        for i in range(n):
            # If line already has a scheme, skip
            if scheme[i]:
                continue
                
            # Assign new letter
            scheme[i] = chr(65 + next_letter)  # 'A', 'B', etc.
            next_letter += 1
            
            # Find all rhyming lines
            for j in range(i+1, n):
                if rhyme_matrix[i][j] >= threshold:
                    scheme[j] = scheme[i]
        
        # Calculate overall rhyme score
        # Count how many lines follow a pattern vs standalone
        pattern_lines = 0
        line_groups = {}
        for letter in scheme:
            if letter not in line_groups:
                line_groups[letter] = 0
            line_groups[letter] += 1
        
        for group_size in line_groups.values():
            if group_size > 1:
                pattern_lines += group_size
        
        score = pattern_lines / n if n > 0 else 0
        
        return {
            'score': score,
            'scheme': ''.join(scheme),
            'pattern': scheme
        }

# Function to load and use the model for lyrics generation
def load_model(model_path):
    """Load the fine-tuned model and tokenizer"""
    tokenizer = GPT2Tokenizer.from_pretrained(model_path)
    model = GPT2LMHeadModel.from_pretrained(model_path)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    return model, tokenizer

def generate_lyrics_with_features(model, tokenizer, audio_features, 
                                 max_length=356, temperature=1.0,
                                 top_k=50, top_p=0.95, num_return_sequences=1):
    """Generate lyrics based on audio features with specified parameters"""
    
    # Create prompt from features
    prompt = "[FEATURES] "
    for key, value in audio_features.items():
        prompt += f"{key}: {value}, "
    prompt += "[LYRICS]"
    
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    # Move to model's device
    device = model.device
    input_ids = input_ids.to(device)
    
    # Generate lyrics
    output_sequences = model.generate(
        input_ids=input_ids,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.pad_token_id
    )
    
    generated_lyrics = []
    for sequence in output_sequences:
        text = tokenizer.decode(sequence, skip_special_tokens=False)
        
        # Extract lyrics part
        lyrics_match = re.search(r'\[LYRICS\](.*)', text, re.DOTALL)
        if lyrics_match:
            lyrics = lyrics_match.group(1).strip()
            generated_lyrics.append(lyrics)
        else:
            generated_lyrics.append(text)
    
    return generated_lyrics

def evaluate_generated_lyrics(lyrics, evaluator=None):
    """Evaluate the quality of generated lyrics"""
    if evaluator is None:
        evaluator = HindiRhymeEvaluator()
    
    # Get rhyme evaluation
    rhyme_eval = evaluator.evaluate_rhyme_scheme(lyrics)
    
    # Count number of lines and words
    lines = [line.strip() for line in lyrics.split('\n') if line.strip()]
    num_lines = len(lines)
    words = []
    for line in lines:
        words.extend(word_tokenize(line))
    num_words = len(words)
    
    # Calculate average line length
    avg_line_length = num_words / num_lines if num_lines > 0 else 0
    
    return {
        'rhyme_score': rhyme_eval['score'],
        'rhyme_scheme': rhyme_eval['scheme'],
        'num_lines': num_lines,
        'num_words': num_words,
        'avg_line_length': avg_line_length
    }

# Example usage script
if __name__ == "__main__":
    # Load model
    model_path = "./hindi_lyrics_gpt2"  # Replace with your model path
    model, tokenizer = load_model(model_path)
    
    # Define sample audio features
   
    sample_features = {
        "duration_ms": 303577,
        "tempo": 112.5,
        "energy": 0.1343526, 
        "loudness": -66.16268,
        "danceability": 0.1596686712,
        "speechiness": 19.04962,
        "acousticness": 0.2,
        "instrumentalness": 0.01,
        "liveness": 0.15,
        "valence": 22.6,
        "rhyme_scheme": "AABB"  # Target rhyme scheme
    }
    # Generate lyrics
    print("Generating lyrics...")
    lyrics_list = generate_lyrics_with_features(
        model, 
        tokenizer, 
        sample_features,
        num_return_sequences=3,  # Generate multiple variations
        temperature=0.7  # Higher temperature for more creative outputs
    )
    
    # Evaluate generated lyrics
    evaluator = HindiRhymeEvaluator()
    
    print("\n===== GENERATED LYRICS =====")
    for i, lyrics in enumerate(lyrics_list):
        print(f"\n----- Lyrics Variation {i+1} -----")
        print(lyrics)
        
        # Evaluate
        evaluation = evaluate_generated_lyrics(lyrics, evaluator)
        print("\nEvaluation:")
        print(f"Rhyme Score: {evaluation['rhyme_score']:.2f}")
        print(f"Rhyme Scheme: {evaluation['rhyme_scheme']}")
        print(f"Number of Lines: {evaluation['num_lines']}")
        print(f"Number of Words: {evaluation['num_words']}")
        print(f"Avg Line Length: {evaluation['avg_line_length']:.2f} words")

## Attempt-2

## Pre-Processing

In [1]:
import re
import unicodedata
import pandas as pd

def preprocess_lyrics(text):
    """
    Comprehensive text preprocessing for Hindi lyrics
    
    Args:
        text (str): Input lyrics text
    
    Returns:
        str: Cleaned and normalized lyrics
    """
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKD', text)
    
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove special characters and numbers (optional, depends on your use case)
    text = re.sub(r'[^a-zA-Zà-ÿ\u0900-\u097F\s]', '', text)
    
    # Lowercase conversion (optional, depends on language)
    # For Hindi, you might want to be careful with lowercasing
    # text = text.lower()
    
    return text

def normalize_audio_features(data):
    """
    Normalize audio features using Min-Max scaling
    
    Args:
        data (pd.DataFrame): DataFrame with audio features
    
    Returns:
        pd.DataFrame: Normalized DataFrame
    """
    # List of features to normalize
    features_to_normalize = [
        'duration_ms', 'energy', 'loudness', 
        'danceability', 'speechiness', 'acousticness', 
        'instrumentalness', 'liveness', 'valence'
    ]
    
    # Create a copy of the dataframe
    normalized_data = data.copy()
    # normalized_data['tempo'] = normalized_data['tempo'].str.strip('[]').apply(lambda x: int(x.strip("'")))    # Apply Min-Max normalization
    
    for feature in features_to_normalize:
        if feature in normalized_data.columns:
            min_val = normalized_data[feature].min()
            max_val = normalized_data[feature].max()
            
            # Avoid division by zero
            if min_val != max_val:
                # print(normalized_data[feature])
                # print(f"max value: {max_val}")
                normalized_data[feature] = (normalized_data[feature] - min_val) / (max_val - min_val)
            else:
                normalized_data[feature] = 0
    
    return normalized_data

def preprocess_dataset(data_path):
    """
    Full preprocessing pipeline for the lyrics dataset
    
    Args:
        data_path (str): Path to the CSV file
    
    Returns:
        pd.DataFrame: Preprocessed and normalized dataset
    """
    # Load the data
    data = pd.read_csv(data_path)
    
    # Normalize audio features
    data = normalize_audio_features(data)
    
    # Preprocess lyrics
    data['Hindi Lyrics'] = data['Hindi Lyrics'].apply(preprocess_lyrics)
    
    return data

# Modify the train_lyrics_model function to use preprocessing
def train_lyrics_model_with_preprocessing(data_path, output_dir, **kwargs):
    """
    Train lyrics model with comprehensive preprocessing
    """
    # Preprocess the dataset
    preprocessed_data = preprocess_dataset(data_path)
    
    # Continue with the original training process
    return train_lyrics_model(
        preprocessed_data,  # Pass preprocessed DataFrame directly
        output_dir,
        **kwargs
    )

## Attempt-3 (without explicit rhyming scheme in prompts)

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import re
import json
from collections import defaultdict

# Custom dataset for lyrics generation
class LyricsDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.inputs = []
        self.max_length = max_length
        
        for _, row in data.iterrows():
            # Create prompt with audio features
            prompt = f"[FEATURES] tempo: {row['tempo']}, "            #without duration_ms
            prompt += f"energy: {row['energy']}, loudness: {row['loudness']}, "
            prompt += f"danceability: {row['danceability']}, speechiness: {row['speechiness']}, "
            prompt += f"acousticness: {row['acousticness']}, instrumentalness: {row['instrumentalness']}, "
            prompt += f"liveness: {row['liveness']}, valence: {row['valence']}, "
            prompt += f"Sentiments: {row['Sentiment']}, "
            
            prompt += "[LANG: ROMANIZED_HINDI] [LYRICS] "
            
            # Complete text (prompt + lyrics)
            complete_text = prompt + row['Hindi Lyrics']
            
            encodings_dict = tokenizer(complete_text, 
                                      truncation=True,
                                      max_length=self.max_length, 
                                      padding="max_length")
            
            self.inputs.append({
                'input_ids': torch.tensor(encodings_dict['input_ids']),
                'attention_mask': torch.tensor(encodings_dict['attention_mask']),
                'labels': torch.tensor(encodings_dict['input_ids'])
            })
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return self.inputs[idx]

# Custom trainer with semantic coherence improvements
class LyricsTrainer:
    def __init__(self, model, tokenizer, train_dataloader, val_dataloader=None, 
                 semantic_weight=0.3, num_epochs=5, lr=5e-5, warmup_steps=0):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.semantic_weight = semantic_weight
        self.num_epochs = num_epochs
        
        # Setup optimizer and scheduler
        self.optimizer = AdamW(model.parameters(), lr=lr)
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer, 
            num_warmup_steps=warmup_steps,
            num_training_steps=len(train_dataloader) * num_epochs
        )
        
        # Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
    
    def _calculate_semantic_coherence(self, generated_text):
        """
        Calculate a basic semantic coherence score
        This is a simplified approach - in practice, you'd use more advanced NLP techniques
        """
        # Extract only the lyrics part (after [LYRICS])
        lyrics_match = re.search(r'\[LYRICS\](.*)', generated_text, re.DOTALL)
        if not lyrics_match:
            return 0.0
        
        lyrics = lyrics_match.group(1).strip()
        
        # Split lyrics into lines
        lines = [line.strip() for line in lyrics.split('\n') if line.strip()]
        if len(lines) <= 1:
            return 0.0
        
        # Basic coherence check: 
        # 1. Minimum word count per line
        # 2. Avoid excessive repetition
        coherence_score = 0.0
        
        # Check word count
        word_count_score = sum(len(line.split()) >= 4 for line in lines) / len(lines)
        coherence_score += word_count_score
        
        # Check for repetition
        unique_lines = len(set(lines)) / len(lines)
        coherence_score += unique_lines
        
        # Normalize score
        return min(max(coherence_score / 2, 0), 1)
    
    def train(self):
        self.model.train()
        total_loss = 0
        
        for epoch in range(self.num_epochs):
            print(f"Starting epoch {epoch+1}/{self.num_epochs}")
            
            for batch_idx, batch in enumerate(self.train_dataloader):
                # Move batch to device
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                # Forward pass
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                # Standard language modeling loss
                lm_loss = outputs.loss
                
                # Semantic coherence loss
                if self.semantic_weight > 0:
                    semantic_loss = torch.tensor(0.0).to(self.device)
                    
                    # For a small batch sample, decode and evaluate semantic coherence
                    if batch_idx % 100 == 0:
                        sample_idx = np.random.randint(0, input_ids.shape[0])
                        sample_text = self.tokenizer.decode(input_ids[sample_idx])
                        semantic_penalty = 1 - self._calculate_semantic_coherence(sample_text)
                        semantic_loss = torch.tensor(semantic_penalty).to(self.device)
                        
                        print(f"Sample semantic penalty: {semantic_penalty:.4f}")
                else:
                    semantic_loss = torch.tensor(0.0).to(self.device)
                
                # Combined loss
                loss = lm_loss + (self.semantic_weight * semantic_loss)
                
                # Backward pass and optimization
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
                
                total_loss += loss.item()
                
                if batch_idx % 50 == 0:
                    print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}")
        
        return total_loss / (self.num_epochs * len(self.train_dataloader))
    
    def save_model(self, output_dir):
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")
    
    def evaluate(self):
        if not self.val_dataloader:
            return None
            
        self.model.eval()
        eval_loss = 0
        
        with torch.no_grad():
            for batch in self.val_dataloader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                eval_loss += outputs.loss.item()
                
        return eval_loss / len(self.val_dataloader)

# Main training function
def train_lyrics_model(data_path, output_dir, model_name="gpt2", 
                       batch_size=4, epochs=5, semantic_weight=0.3):
    """Train GPT-2 model on lyrics dataset with semantic coherence improvements"""
    
    # Load and prepare data
    data = data_path
    
    # Make sure all required columns exist
    required_columns = [
        'tempo', 'energy', 'loudness', 'danceability',
        'speechiness', 'acousticness', 'instrumentalness', 'liveness',
        'valence', 'Hindi Lyrics', 'Sentiment'
    ]
    
    for col in required_columns:
        if col not in data.columns:
            print(f"Warning: Missing column {col}")
            if col == 'Hindi Lyrics':  # Lyrics column is essential
                raise ValueError("Hindi Lyrics column is missing!")
            elif col != 'Sentiment':
                data[col] = 0  # Fill with defaults for missing audio features
            else:
                data[col] = 'neutral'  # Default sentiment
    
    # Split data
    train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)
    
    # Load tokenizer and model
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name)
    
    # Add special tokens
    special_tokens = {
        'pad_token': '[PAD]',
        'additional_special_tokens': ['[FEATURES]', '[LYRICS]', '[LANG: ROMANIZED_HINDI]'],
    }
    tokenizer.add_special_tokens(special_tokens)
    model.resize_token_embeddings(len(tokenizer))
    
    # Create datasets
    train_dataset = LyricsDataset(train_data, tokenizer)
    val_dataset = LyricsDataset(val_data, tokenizer)
    
    # Create dataloaders
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size
    )
    
    # Set up trainer
    trainer = LyricsTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        semantic_weight=semantic_weight,
        num_epochs=epochs
    )
    
    # Train model
    train_loss = trainer.train()
    print(f"Training completed with average loss: {train_loss:.4f}")
    
    # Evaluate model
    eval_loss = trainer.evaluate()
    if eval_loss:
        print(f"Evaluation loss: {eval_loss:.4f}")
    
    # Save model
    trainer.save_model(output_dir)
    
    return model, tokenizer

# Function to generate lyrics based on audio features
def generate_lyrics(model, tokenizer, audio_features, max_length=200, 
                   temperature=0.7, top_k=40, top_p=0.9):
    """Generate lyrics based on provided audio features"""
    
    # Create prompt from features
    
    prompt = "[FEATURES] "
    for key, value in audio_features.items():
        prompt += f"{key}: {value}, "
    prompt += "Generate lyrics with above features"
    prompt += "[LYRICS]"
    
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    # Move to model's device
    device = model.device
    input_ids = input_ids.to(device)
    
    # Set attention mask
    attention_mask = torch.ones(input_ids.shape, device=device)
    
    # Generate lyrics
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=2,  # Prevent immediate repetition
        num_return_sequences=3   # Generate multiple alternatives
    )
    
    # Decode outputs and filter
    generated_texts = []
    for seq in output:
        generated_text = tokenizer.decode(seq, skip_special_tokens=False)
        
        # Extract only the lyrics part
        lyrics_match = re.search(r'\[LYRICS\](.*)', generated_text, re.DOTALL)
        if lyrics_match:
            lyrics = lyrics_match.group(1).strip()
            
            # Basic filtering for meaningful lyrics
            if len(lyrics.split()) > 20:  # Minimum word count
                generated_texts.append(lyrics)
    
    # Return the most semantically coherent text
    return generated_texts[0] if generated_texts else "Could not generate meaningful lyrics."

# Example usage
if __name__ == "__main__":
    # This would be your main execution code
    data_path = "/kaggle/input/final-hindi-dataset/hindi_songs_with_sentiment.csv"
    output_dir = "./hindi_lyrics_gpt2"

    normalized_data = preprocess_dataset(data_path)
    # Train model
    model, tokenizer = train_lyrics_model(
        data_path=normalized_data,
        output_dir=output_dir,
        semantic_weight=0.3,  # Weight for the semantic coherence component of the loss
        epochs=5
    )
    
    # Example of generating lyrics
    sample_features = {
        "tempo": 112.5,
        "energy": 0.1343526, 
        "loudness": -66.16268,
        "danceability": 0.1596686712,
        "speechiness": 19.04962,
        "acousticness": 0.2,
        "instrumentalness": 0.01,
        "liveness": 0.15,
        "valence": 22.6,
    }
    
    generated_lyrics = generate_lyrics(model, tokenizer, sample_features)
    print("Generated Lyrics:")
    print(generated_lyrics)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Starting epoch 1/8
Sample semantic penalty: 1.0000
Epoch 1, Batch 0, Loss: 5.7453
Epoch 1, Batch 50, Loss: 3.3913
Sample semantic penalty: 1.0000
Epoch 1, Batch 100, Loss: 3.1485
Epoch 1, Batch 150, Loss: 2.4455
Sample semantic penalty: 1.0000
Epoch 1, Batch 200, Loss: 3.2889
Epoch 1, Batch 250, Loss: 2.5049
Sample semantic penalty: 1.0000
Epoch 1, Batch 300, Loss: 2.6704
Epoch 1, Batch 350, Loss: 2.3086
Sample semantic penalty: 1.0000
Epoch 1, Batch 400, Loss: 2.4369
Epoch 1, Batch 450, Loss: 2.2774
Sample semantic penalty: 1.0000
Epoch 1, Batch 500, Loss: 2.5090
Starting epoch 2/8
Sample semantic penalty: 1.0000
Epoch 2, Batch 0, Loss: 2.3361
Epoch 2, Batch 50, Loss: 2.0799
Sample semantic penalty: 1.0000
Epoch 2, Batch 100, Loss: 2.4549
Epoch 2, Batch 150, Loss: 2.1237
Sample semantic penalty: 1.0000
Epoch 2, Batch 200, Loss: 2.3722
Epoch 2, Batch 250, Loss: 2.3385
Sample semantic penalty: 1.0000
Epoch 2, Batch 300, Loss: 2.3064
Epoch 2, Batch 350, Loss: 2.1917
Sample semantic penal

In [12]:
# Function to generate lyrics based on audio features
def generate_lyrics(model, tokenizer, audio_features, max_length=256, 
                   temperature=0.5, top_k=40, top_p=0.9):
    """Generate lyrics based on provided audio features"""
    
    # Create prompt from features
    prompt = "[FEATURES] "
    for key, value in audio_features.items():
        prompt += f"{key}: {value}, "
    prompt += "Generate hindi lyrics linewise in structured way"
    prompt += "[LANG: ROMANIZED_HINDI] [LYRICS]"
    
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    # Move to model's device
    device = model.device
    input_ids = input_ids.to(device)
    
    # Set attention mask
    attention_mask = torch.ones(input_ids.shape, device=device)
    
    # Generate lyrics
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        no_repeat_ngram_size=2,  # Prevent immediate repetition
        num_return_sequences=3   # Generate multiple alternatives
    )
    
    # Decode outputs and filter
    generated_texts = []
    for seq in output:
        generated_text = tokenizer.decode(seq, skip_special_tokens=False)
        
        # Extract only the lyrics part
        lyrics_match = re.search(r'\[LYRICS\](.*)', generated_text, re.DOTALL)
        if lyrics_match:
            lyrics = lyrics_match.group(1).strip()
            
            # Basic filtering for meaningful lyrics
            if len(lyrics.split()) > 20:  # Minimum word count
                generated_texts.append(lyrics)
    
    # Return the most semantically coherent text
    return generated_texts[0] if generated_texts else "Could not generate meaningful lyrics."
    
# Example of generating lyrics
sample_features = {
"duration_ms": 770005,
"tempo": 112.5,
"energy": 0.1343526, 
"loudness": -66.16268,
"danceability": 0.1596686712,
"speechiness": 19.04962,
"acousticness": 0.2,
"instrumentalness": 0.01,
"liveness": 0.15,
"valence": 22.6,
"Sentiment": "Sad"
}

generated_lyrics = generate_lyrics(model, tokenizer, sample_features)
print("Generated Lyrics:")
print(generated_lyrics)

Generated Lyrics:
Lyrics Dil ki dhadkan ki koi jaaneman Dil kehna hai dil kehti hain Dil hawaon mein aaya haan haal haa ha Dil mehka haath haaye dil mehar meherbaan Dil pehle haathon megha haai haar haale Dil se kya haara haari haare Dil kaisa haati haat haala haalo haali Dil yeh kahani haq haaris mehenge haule hail haile Dil kyun hazaar mehaas haakar  Dil ne dil kaise dil ko kahan Dilne dil ki hawayein dil ne kaha haate haata hale Dil joh aankhon


## mt-5 model experiment(Need compute to run this)

In [ ]:
import torch
import numpy as np
import pandas as pd
import re
import unicodedata
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
from transformers import AdamW, get_linear_schedule_with_warmup

# Download necessary NLTK resources
try:
    nltk.download('punkt')
    nltk.download('stopwords')
except:
    print("Could not download NLTK resources. Some preprocessing might be limited.")

class LyricsPreprocessor:
    def __init__(self, language='english'):
        """
        Initialize preprocessor with language-specific settings
        
        Args:
            language (str): Language of the text to preprocessA
        """
        self.stopwords = set(stopwords.words(language)) if language in stopwords._fileids else set()
    
    def clean_text(self, text):
        """
        Comprehensive text cleaning method
        
        Args:
            text (str): Input text to clean
        
        Returns:
            str: Cleaned text
        """
        if not isinstance(text, str):
            return ""
        
        # Normalize Unicode characters
        text = unicodedata.normalize('NFKD', text)
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove special characters and numbers
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        
        # Remove extra whitespaces
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def remove_stopwords(self, text):
        """
        Remove stopwords from text
        
        Args:
            text (str): Input text
        
        Returns:
            str: Text with stopwords removed
        """
        # Tokenize the text
        words = text.split()
        
        # Remove stopwords
        filtered_words = [word for word in words if word not in self.stopwords]
        
        return ' '.join(filtered_words)
    
    def preprocess_lyrics(self, lyrics, remove_stops=True):
        """
        Comprehensive lyrics preprocessing
        
        Args:
            lyrics (str): Input lyrics
            remove_stops (bool): Whether to remove stopwords
        
        Returns:
            str: Preprocessed lyrics
        """
        # Clean the text
        cleaned_text = self.clean_text(lyrics)
        
        # Optionally remove stopwords
        if remove_stops:
            cleaned_text = self.remove_stopwords(cleaned_text)
        
        return cleaned_text

class LyricsDataset(Dataset):
    def __init__(self, data, tokenizer, preprocessor, max_length=512):
        """
        Custom Dataset for Lyrics Generation
        
        Args:
            data (pd.DataFrame): Input dataframe
            tokenizer (MT5Tokenizer): Tokenizer for encoding
            preprocessor (LyricsPreprocessor): Text preprocessor
            max_length (int): Maximum sequence length
        """
        self.tokenizer = tokenizer
        self.preprocessor = preprocessor
        self.max_length = max_length
        self.inputs = []
        
        for _, row in data.iterrows():
            # Preprocess lyrics
            preprocessed_lyrics = preprocessor.preprocess_lyrics(row['Hindi Lyrics'])
            
            # Create input features string
            features_str = (
                f"duration: {row.get('duration_ms', 0)}, "
                f"tempo: {row.get('tempo', 0)}, "
                f"energy: {row.get('energy', 0)}, "
                f"loudness: {row.get('loudness', 0)}, "
                f"danceability: {row.get('danceability', 0)}, "
                f"valence: {row.get('valence', 0)}",
                f"Sentiments: {row.get('Sentiment', 0)}"
            )
            
            # Combine features and lyrics
            full_text = f"Features: {features_str} Lyrics: {preprocessed_lyrics}"
            
            # Tokenize
            encodings = tokenizer(
                full_text, 
                truncation=True, 
                max_length=self.max_length, 
                padding="max_length"
            )
            
            self.inputs.append({
                'input_ids': torch.tensor(encodings['input_ids']),
                'attention_mask': torch.tensor(encodings['attention_mask']),
                'labels': torch.tensor(encodings['input_ids'])
            })
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return self.inputs[idx]

class MT5LyricsTrainer:
    def __init__(self, model, tokenizer, train_dataloader, 
                 val_dataloader=None, lr=5e-5, epochs=3):
        """
        Custom Trainer for mT5 Lyrics Generation
        
        Args:
            model (MT5ForConditionalGeneration): mT5 model
            tokenizer (MT5Tokenizer): Tokenizer
            train_dataloader (DataLoader): Training data loader
            val_dataloader (DataLoader, optional): Validation data loader
            lr (float): Learning rate
            epochs (int): Number of training epochs
        """
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.epochs = epochs
        
        # Optimizer and scheduler
        self.optimizer = AdamW(model.parameters(), lr=lr)
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=0,
            num_training_steps=len(train_dataloader) * epochs
        )
        
        # Device setup
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
    
    def train(self):
        """
        Train the mT5 model
        
        Returns:
            float: Average training loss
        """
        self.model.train()
        total_loss = 0
        
        for epoch in range(self.epochs):
            print(f"Epoch {epoch + 1}/{self.epochs}")
            
            for batch_idx, batch in enumerate(self.train_dataloader):
                # Move to device
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                # Zero gradients
                self.optimizer.zero_grad()
                
                # Forward pass
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                loss = outputs.loss
                total_loss += loss.item()
                
                # Backward pass
                loss.backward()
                
                # Clip gradients
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                
                # Optimizer step
                self.optimizer.step()
                self.scheduler.step()
                
                # Logging
                if batch_idx % 50 == 0:
                    print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")
            
            # Evaluation after each epoch
            if self.val_dataloader:
                val_loss = self.evaluate()
                print(f"Validation Loss: {val_loss:.4f}")
        
        return total_loss / (len(self.train_dataloader) * self.epochs)
    
    def evaluate(self):
        """
        Evaluate the model on validation data
        
        Returns:
            float: Validation loss
        """
        if not self.val_dataloader:
            return 0.0
        
        self.model.eval()
        total_val_loss = 0
        
        with torch.no_grad():
            for batch in self.val_dataloader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                
                total_val_loss += outputs.loss.item()
        
        return total_val_loss / len(self.val_dataloader)
    
    def save_model(self, output_dir):
        """
        Save the trained model and tokenizer
        
        Args:
            output_dir (str): Directory to save model
        """
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

def train_mt5_lyrics_model(
    data_path, 
    output_dir, 
    model_name="google/mt5-small", 
    batch_size=4, 
    epochs=3
):
    """
    Train mT5 model for lyrics generation
    
    Args:
        data_path (str): Path to input CSV
        output_dir (str): Directory to save model
        model_name (str): Pre-trained mT5 model to use
        batch_size (int): Training batch size
        epochs (int): Number of training epochs
    
    Returns:
        tuple: Trained model and tokenizer
    """
    # Load and preprocess data
    data = pd.read_csv(data_path)
    
    # Ensure required columns exist
    required_columns = [
        'duration_ms', 'tempo', 'energy', 'loudness', 'danceability',
        'valence', 'Hindi Lyrics', 'Sentiment'
    ]
    
    for col in required_columns:
        if col not in data.columns:
            print(f"Warning: Missing column {col}")
            if col == 'Hindi Lyrics':
                raise ValueError("Lyrics column is required!")
            else:
                # Fill missing feature columns with default values
                data[col] = 0
    
    # Split data
    train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)
    
    # Initialize preprocessor and tokenizer
    preprocessor = LyricsPreprocessor()
    tokenizer = MT5Tokenizer.from_pretrained(model_name)
    
    # Create datasets
    train_dataset = LyricsDataset(train_data, tokenizer, preprocessor)
    val_dataset = LyricsDataset(val_data, tokenizer, preprocessor)
    
    # Create dataloaders
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size
    )
    
    # Load model
    model = MT5ForConditionalGeneration.from_pretrained(model_name)
    
    # Initialize trainer
    trainer = MT5LyricsTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=epochs
    )
    
    # Train model
    train_loss = trainer.train()
    print(f"Training completed with average loss: {train_loss:.4f}")
    
    # Save model
    trainer.save_model(output_dir)
    
    return model, tokenizer

def generate_lyrics(
    model, 
    tokenizer, 
    audio_features, 
    max_length=200, 
    num_return_sequences=3
):
    """
    Generate lyrics based on audio features
    
    Args:
        model (MT5ForConditionalGeneration): Trained model
        tokenizer (MT5Tokenizer): Tokenizer
        audio_features (dict): Dictionary of audio features
        max_length (int): Maximum generation length
        num_return_sequences (int): Number of lyric sequences to generate
    
    Returns:
        list: Generated lyrics
    """
    # Prepare input features string
    features_str = (
        f"duration: {audio_features.get('duration_ms', 0)}, "
        f"tempo: {audio_features.get('tempo', 0)}, "
        f"energy: {audio_features.get('energy', 0)}, "
        f"loudness: {audio_features.get('loudness', 0)}, "
        f"danceability: {audio_features.get('danceability', 0)}, "
        f"valence: {audio_features.get('valence', 0)}"
    )
    
    # Prepare input
    input_text = f"Features: {features_str} Lyrics:"
    
    # Tokenize input
    inputs = tokenizer(
        input_text, 
        return_tensors="pt", 
        max_length=512, 
        truncation=True
    )
    
    # Move to device
    device = next(model.parameters()).device
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    
    # Generate lyrics
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )
    
    # Decode generated lyrics
    generated_lyrics = []
    for output in outputs:
        lyrics = tokenizer.decode(output, skip_special_tokens=True)
        generated_lyrics.append(lyrics)
    
    return generated_lyrics

# Example usage
if __name__ == "__main__":
    # Path to your CSV file
    data_path = "/kaggle/input/final-hindi-dataset/hindi_songs_with_sentiment.csv"
    output_dir = "./mt5_lyrics_model"
    
    # Train the model
    model, tokenizer = train_mt5_lyrics_model(
        data_path=data_path,
        output_dir=output_dir,
        model_name="google/mt5-small",
        batch_size=4,
        epochs=3
    )
    
    # Example audio features for generation
    # Example of generating lyrics
    sample_features = {
    "duration_ms": 303577,
    "tempo": 112.5,
    "energy": 0.1343526, 
    "loudness": -66.16268,
    "danceability": 0.1596686712,
    "speechiness": 19.04962,
    "acousticness": 0.2,
    "instrumentalness": 0.01,
    "liveness": 0.15,
    "valence": 22.6,
    }
    
    # Generate lyrics
    generated_lyrics = generate_lyrics(model, tokenizer, sample_features)
    
    # Print generated lyrics
    print("Generated Lyrics:")
    for i, lyrics in enumerate(generated_lyrics, 1):
        print(f"\nLyrics {i}:")
        print(lyrics)